In [13]:
import csv
import json
import librosa
import shutil
import numpy as np
from pathlib import Path
from librosa.feature import melspectrogram

In [14]:
# Constants for Mel-Spectrogram preprocessing
LABEL_MAP = {
    "Accordion": 0,
    "AcousticGuitar": 1,
    "Banjo": 2,
    "BassGuitar": 3,
    "Clarinet": 4,
    "Cowbell": 5,
    "Cymbals": 6,
    "Dobro": 7,
    "DrumSet": 8,
    "ElectricGuitar": 9,
    "FloorTom": 10,
    "Flute": 11,
    "Harmonica": 12,
    "Harmonium": 13,
    "HiHats": 14,
    "Horn": 15,
    "Keyboard": 16,
    "Mandolin": 17,
    "Organ": 18,
    "Piano": 19,
    "Saxophone": 20,
    "Shakers": 21,
    "Tambourine": 22,
    "Trombone": 23,
    "Trumpet": 24,
    "Ukulele": 25,
    "Vibraphone": 26,
    "Violin": 27
}

SAMPLE_RATE = 16000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 320

In [15]:
MEAN_DB = 0
STD_DB = 0

manifest_headers = ["Path", "Label", "Mel_Path"]
manifest_data = []

In [16]:
# Create the processed directory
processed_dir = Path("data/processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)

processed_dir.mkdir(parents=True, exist_ok=True)

total_sum = 0
total_sq_sum = 0
total_elements = 0
max_length = -np.inf
# Create the numpy files
for file in Path("data/raw").rglob("*.wav"):
    try:
        print(f"Processing: {file}")

        waveform, _ = librosa.load(file, sr=SAMPLE_RATE)
        mel_spectrogram = melspectrogram(y=waveform, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

        # Save numpy file
        label = str(file.parent.name).replace("_", " ")
        label = label.title()
        label = label.replace(" ", "")
        output_path = processed_dir / f"{label}_{file.stem}_mel.npy"
        np.save(output_path, mel_spectrogram_db)

        # Increment metrics
        total_sum += np.sum(mel_spectrogram_db)
        total_sq_sum += np.sum(mel_spectrogram_db ** 2)
        total_elements += mel_spectrogram_db.size

        if mel_spectrogram.shape[1] > max_length:
            max_length = mel_spectrogram.shape[1]

        # Save data to the manifest
        manifest_data.append([file, label, output_path])

    except Exception as e:
        print(f"Error processing {file}: {e}")


Processing: data\raw\music_dataset\Accordion\100.wav
Processing: data\raw\music_dataset\Accordion\1000.wav
Processing: data\raw\music_dataset\Accordion\1001.wav
Processing: data\raw\music_dataset\Accordion\1002.wav
Processing: data\raw\music_dataset\Accordion\1003.wav
Processing: data\raw\music_dataset\Accordion\1004.wav
Processing: data\raw\music_dataset\Accordion\1005.wav
Processing: data\raw\music_dataset\Accordion\1006.wav
Processing: data\raw\music_dataset\Accordion\1007.wav
Processing: data\raw\music_dataset\Accordion\1008.wav
Processing: data\raw\music_dataset\Accordion\1009.wav
Processing: data\raw\music_dataset\Accordion\101.wav
Processing: data\raw\music_dataset\Accordion\1010.wav
Processing: data\raw\music_dataset\Accordion\1011.wav
Processing: data\raw\music_dataset\Accordion\1012.wav
Processing: data\raw\music_dataset\Accordion\1013.wav
Processing: data\raw\music_dataset\Accordion\1014.wav
Processing: data\raw\music_dataset\Accordion\1015.wav
Processing: data\raw\music_dat

In [17]:
MEAN_DB = total_sum / total_elements
STD_DB = np.sqrt((total_sq_sum / total_elements) - (MEAN_DB ** 2))
MAX_MLEN = max_length

print(f"MEAN_DB: {MEAN_DB:.3f}")
print(f"STD_DB: {STD_DB:.3f}")
print(f"MAX_MLEN: {MAX_MLEN:.3f}")

MEAN_DB: -49.982
STD_DB: 18.420


In [22]:
metadata_dir = processed_dir / "metadata"
if metadata_dir.exists():
    shutil.rmtree(metadata_dir)

metadata_dir.mkdir(parents=True, exist_ok=True)

mel_params = {
    "LABEL_MAP": LABEL_MAP,
    "SAMPLE_RATE": SAMPLE_RATE,
    "N_MELS": N_MELS,
    "N_FFT": N_FFT,
    "HOP_LENGTH": HOP_LENGTH,
    "MEAN_DB": float(MEAN_DB),
    "STD_DB": float(STD_DB),
    "MAX_MLEN": MAX_MLEN,
}

with open(metadata_dir / "mel_params.json", "w") as f:
    json.dump(mel_params, f)

In [23]:
with open(metadata_dir / "manifest.csv", "w", newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(manifest_headers)
    writer.writerows(manifest_data)